In [4]:
import sys
sys.path.append('..')
from utils.prompts import render
from utils.llm_client import LLMClient
from utils.logging_utils import log_llm_call
from utils.router import pick_model

In [5]:
with open('..\\data\\Incidents.txt', 'r') as file:
    lines = file.readlines()

incidents = lines[1:]

reasoning_model = pick_model('openai', 'cot')
client_reasoning = LLMClient('openai', reasoning_model)

print(f'Using reasoning model: {reasoning_model}')

# Step A (CoT Reasoning)
problem="You have limited resources and multiple victims. "

instructions=f"""
You are given disaster incidents.

Scoring Rules:
- Base score: 5
- +2 if any age > 60 or < 5
- +3 if Main Need is 'Rescue'
- +1 if Main Need is 'Insulin' or 'Medicine'

Analyze EACH incident one by one.
Explain reasoning briefly.
Return output exactly as:

Incident <ID> | Area | Score: X/10 | Reason
"""

prompt_text, spec = render(
    'cot_reasoning.v1',
    role="Real-Time Strategic Triage & Resource Coordinator",
    problem=problem
)

full_prompt = f"""text: {incidents}
instructions: {instructions}"""

messageCot=[{'role': 'user', 'content': full_prompt}]
responseCot = client_reasoning.chat(messageCot, temperature=spec.temperature, max_tokens=spec.max_tokens)

print('CoT Response (Multiple Solution Paths):')
print('=' * 80)
print(responseCot['text'])
print('=' * 80)

log_llm_call('openai', reasoning_model, 'cot_reasoning', responseCot['latency_ms'], responseCot['usage'])

Using reasoning model: o3-mini
CoT Response (Multiple Solution Paths):
Incident 1 | Gampaha | Score: 5/10 | Reason: Base score 5; no age bonus because the age range (20–40) does not include anyone over 60 or under 5; Main Need "Water" gives no extra points.
Incident 2 | Ja-Ela | Score: 8/10 | Reason: Base score 5; age 75 qualifies for +2; Main Need "Insulin" provides an extra +1.
Incident 3 | Ragama | Score: 8/10 | Reason: Base score 5; Main Need "Rescue" gives +3; ages (10 and 35) do not trigger the age bonus.


In [6]:
problem = """
You have ONE rescue boat starting at Ragama.

Travel times:
- Ragama → Ja-Ela : 10 minutes
- Ja-Ela → Gampaha : 40 minutes

Boat can resolve ONE incident per stop.
Unlimited capacity.

Goal:
Maximize total priority score saved in minimum time.
"""

instructions = """
Explore THREE branches:

Branch 1: Highest priority score first (Greedy)
Branch 2: Closest location first (Speed)
Branch 3: Furthest location first (Logistics)

For EACH branch:
- Show route
- Total time
- Total score saved

Finally, SELECT the optimal route and justify why.
"""

prompt_text, spec = render(
    'tot_reasoning.v1',
    role="Real-Time Strategic Triage & Resource Coordinator",
    problem=problem,
    branches=3
)

full_prompt = f"""
Incident priority scores (from Step A):
{responseCot['text']}

{instructions}
"""

responseTot = client_reasoning.chat(
    [{'role': 'user', 'content': full_prompt}],
    temperature=spec.temperature,
    max_tokens=spec.max_tokens
)

print("ToT Response (Strategy Selection)")
print("=" * 80)
print(responseTot['text'])
print("=" * 80)

log_llm_call('openai', reasoning_model, 'tot_reasoning', responseTot['latency_ms'], responseTot['usage'])

ToT Response (Strategy Selection)
Below is one way to work through the problem. In our example we have three incidents with these priority scores and locations:

• Incident 1 (“Gampaha”) — Score: 5  
• Incident 2 (“Ja‐Ela”) — Score: 8  
• Incident 3 (“Ragama”) — Score: 8  

We now “explore” three branches. For our analysis we assume that the emergency response team starts at a central headquarters (HQ) in Colombo. (Note that no exact distances or travel times are provided by the problem, so in the following example we assume reasonable travel‐time estimates. The inventional travel–time “matrix” is as follows.)

─────────────────────────────  
Assumed Travel Times (in minutes):

• From HQ to:  
  – Ja‐Ela (Incident 2): 15  
  – Ragama (Incident 3): 20  
  – Gampaha (Incident 1): 25  

• Between incident sites:
  – Ja‐Ela ↔ Ragama: 10  
  – Ja‐Ela ↔ Gampaha: 15  
  – Ragama ↔ Gampaha: 10  
─────────────────────────────  

Also assume that “total score saved” is the sum of the incidents’ 